In [ ]:
# put schema in a text file #todo

import sqlite3
import pandas as pd

# Function to execute an SQL script from a file
def execute_sql_file(db_name, sql_file):
    with open(sql_file, 'r') as file:
        sql_script = file.read()
    
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    try:
        cursor.executescript(sql_script)
        conn.commit()
        print(f"✅ Executed script: {sql_file}")
    except sqlite3.Error as e:
        print(f"❌ Error executing {sql_file}: {e}")
    
    conn.close()

# Function to print database structure
def print_db_structure(db_name, step_description):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    
    print(f"\n=== {step_description} ===")
    
    query = "SELECT name, type, sql FROM sqlite_master WHERE type IN ('table', 'index', 'trigger')"
    df = pd.read_sql(query, conn)
    
    if df.empty:
        print("⚠️ No tables, indexes, or triggers found.")
    else:
        print("📌 Database Structure:")
        display(df)
    
    conn.close()

# Function to display table contents
def show_table(db_name, table_name):
    conn = sqlite3.connect(db_name)
    
    try:
        df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
        print(f"\n📊 Table: {table_name}")
        display(df)
    except Exception as e:
        print(f"⚠️ Error reading table '{table_name}': {e}")
    
    conn.close()


# Database file name
db_file = "mfa_schema_design.db"

# Step 1: Create Initial Database Schema
print("\n=== Step 1: Creating Initial Collections Schema ===")
execute_sql_file(db_file, "Schema/mfa_collections.sql")
print_db_structure(db_file, "Step 1: After Creating Collections Schema")

# Step 2: Load Additional Schema with Data
print("\n=== Step 2: Expanding Schema with Additional Tables and Data ===")
execute_sql_file(db_file, "Schema/mfa_full.sql")
print_db_structure(db_file, "Step 2: After Expanding Schema")

# Show table contents
show_table(db_file, "collections")
show_table(db_file, "artists")
show_table(db_file, "created")

# Step 3: Modify Schema to Add a CHECK Constraint
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

print("\n=== Step 3: Modifying Schema to Add a CHECK Constraint ===")
alter_schema = """
ALTER TABLE collections ADD COLUMN year_acquired INTEGER CHECK(year_acquired >= 1800);
"""

try:
    cursor.execute(alter_schema)
    conn.commit()
    print("✅ Added CHECK constraint on 'year_acquired'")
except sqlite3.Error as e:
    print(f"❌ Error modifying schema: {e}")

conn.close()
print_db_structure(db_file, "Step 3: After Adding CHECK Constraint")

# Step 4: Test Constraint - Insert Invalid Data
print("\n=== Step 4: Testing CHECK Constraint ===")
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

try:
    cursor.execute("""
        INSERT INTO collections (title, accession_number, acquired, year_acquired) 
        VALUES ('Test Art', '99.9999', '2023-01-01', 1700);
    """)
    conn.commit()
    print("✅ Inserted data successfully")
except sqlite3.Error as e:
    print(f"⚠️ Constraint violation: {e}")

conn.close()
print_db_structure(db_file, "Step 4: Final Database Structure")

# Show updated table contents
show_table(db_file, "collections")

